# DDoS Detector Training

Training notebook placeholder.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
file_path = '/content/drive/MyDrive/spectra/ddo_mon_Jun_14.csv'
df1 = pd.read_csv(file_path)

file_path = '/content/drive/MyDrive/spectra/ddo_tue_jun_15.csv'
df2 = pd.read_csv(file_path)

file_path = '/content/drive/MyDrive/spectra/ddo_sat_jun_12.csv'
df3 = pd.read_csv(file_path)

file_path = '/content/drive/MyDrive/spectra/ddo_sun_jun_13.csv'
df4 = pd.read_csv(file_path)

file_path = '/content/drive/MyDrive/spectra/ddo_wed_jun_16.csv'
df5 = pd.read_csv(file_path)

file_path = '/content/drive/MyDrive/spectra/ddo_thu_hun_17.csv'
df6 = pd.read_csv(file_path)

In [5]:
df = pd.concat([df1, df2, df3, df4, df5, df6], ignore_index=True)
df.info()
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2071657 entries, 0 to 2071656
Data columns (total 21 columns):
 #   Column                          Dtype 
---  ------                          ----- 
 0   generated                       object
 1   appName                         object
 2   totalSourceBytes                int64 
 3   totalDestinationBytes           int64 
 4   totalDestinationPackets         int64 
 5   totalSourcePackets              int64 
 6   sourcePayloadAsBase64           object
 7   sourcePayloadAsUTF              object
 8   destinationPayloadAsBase64      object
 9   destinationPayloadAsUTF         object
 10  direction                       object
 11  sourceTCPFlagsDescription       object
 12  destinationTCPFlagsDescription  object
 13  source                          object
 14  protocolName                    object
 15  sourcePort                      int64 
 16  destination                     object
 17  destinationPort                 int64 
 18  st

,generated,appName,totalSourceBytes,totalDestinationBytes,totalDestinationPackets,totalSourcePackets,sourcePayloadAsBase64,sourcePayloadAsUTF,destinationPayloadAsBase64,destinationPayloadAsUTF,...,sourceTCPFlagsDescription,destinationTCPFlagsDescription,source,protocolName,sourcePort,destination,destinationPort,startDateTime,stopDateTime,Label
0,3/11/2014 18:21,Unknown_UDP,16076,0,0,178,NaN,NaN,NaN,NaN,...,NaN,NaN,192.168.5.122,udp_ip,5353,224.0.0.251,5353,6/13/2010 23:57,6/14/2010 0:11,Normal
1,3/11/2014 18:21,HTTPImageTransfer,384,0,0,6,NaN,NaN,NaN,NaN,...,"F,A",NaN,192.168.2.111,tcp_ip,4435,206.217.198.186,80,6/13/2010 23:58,6/14/2010 0:01,Normal
2,3/11/2014 18:21,DNS,171,642,4,2,NaN,NaN,NaN,NaN,...,NaN,NaN,192.168.4.119,udp_ip,4428,192.168.5.122,53,6/13/2010 23:58,6/13/2010 23:59,Normal
3,3/11/2014 18:21,HTTPImageTransfer,384,0,0,6,NaN,NaN,NaN,NaN,...,"F,A",NaN,192.168.4.119,tcp_ip,3639,219.94.203.105,80,6/13/2010 23:58,6/14/2010 0:00,Normal
4,3/11/2014 18:21,HTTPImageTransfer,186,128,2,2,NaN,NaN,NaN,NaN,...,"F,P,A",R,192.168.4.119,tcp_ip,3641,98.137.80.50,80,6/13/2010 23:58,6/13/2010 23:59,Normal


In [6]:
df['Label'].value_counts()

,count
Label,
Normal,2002747
Attack,68910


In [7]:
drop_cols = [
    'sourcePayloadAsBase64', 'sourcePayloadAsUTF',
    'destinationPayloadAsBase64', 'destinationPayloadAsUTF',
    'generated', 'appName'
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

In [8]:
df['startDateTime'] = pd.to_datetime(df['startDateTime'])
df['stopDateTime'] = pd.to_datetime(df['stopDateTime'])
df = df.sort_values('startDateTime').reset_index(drop=True)

In [9]:
df['duration'] = (df['stopDateTime'] - df['startDateTime']).dt.total_seconds().clip(lower=0.001)

df['packets_per_second'] = (df['totalSourcePackets'] + df['totalDestinationPackets']) / df['duration']
df['bytes_per_second'] = (df['totalSourceBytes'] + df['totalDestinationBytes']) / df['duration']

# Traffic asymmetry features
df['byte_asymmetry'] = np.abs(df['totalSourceBytes'] - df['totalDestinationBytes']) / (df['totalSourceBytes'] + df['totalDestinationBytes'] + 1)
df['packet_asymmetry'] = np.abs(df['totalSourcePackets'] - df['totalDestinationPackets']) / (df['totalSourcePackets'] + df['totalDestinationPackets'] + 1)
df['bidirectional_ratio'] = df['totalDestinationBytes'] / (df['totalSourceBytes'] + df['totalDestinationBytes'] + 1)

# TCP flag features
df['flags'] = df['sourceTCPFlagsDescription'].fillna('')
df['syn_ratio'] = df['flags'].str.contains('S').astype(int)
df['rst_ratio'] = df['flags'].str.contains('R').astype(int)
df['syn_without_data'] = ((df['syn_ratio'] == 1) & (df['totalSourceBytes'] < 100)).astype(int)

df = df.set_index('startDateTime')

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2071657 entries, 2010-06-11 20:10:00 to 2010-06-17 23:58:00
Data columns (total 24 columns):
 #   Column                          Dtype         
---  ------                          -----         
 0   totalSourceBytes                int64         
 1   totalDestinationBytes           int64         
 2   totalDestinationPackets         int64         
 3   totalSourcePackets              int64         
 4   direction                       object        
 5   sourceTCPFlagsDescription       object        
 6   destinationTCPFlagsDescription  object        
 7   source                          object        
 8   protocolName                    object        
 9   sourcePort                      int64         
 10  destination                     object        
 11  destinationPort                 int64         
 12  stopDateTime                    datetime64[ns]
 13  Label                           object        
 14  duration         

In [11]:
df.describe()

,totalSourceBytes,totalDestinationBytes,totalDestinationPackets,totalSourcePackets,sourcePort,destinationPort,stopDateTime,duration,packets_per_second,bytes_per_second,byte_asymmetry,packet_asymmetry,bidirectional_ratio,syn_ratio,rst_ratio,syn_without_data
count,2.071657e+06,2.071657e+06,2.071657e+06,2.071657e+06,2.071657e+06,2.071657e+06,2071657,2.071657e+06,2.071657e+06,2.071657e+06,2.071657e+06,2.071657e+06,2.071657e+06,2.071657e+06,2.071657e+06,2.071657e+06
mean,2.460947e+03,3.448911e+04,3.030068e+01,1.984019e+01,1.413759e+04,1.882091e+03,2010-06-15 14:18:37.406172928,4.898777e+01,2.521548e+04,1.633100e+07,6.159376e-01,1.540684e-01,7.049374e-01,6.964782e-01,2.139833e-02,1.152990e-02
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2010-06-11 23:59:00,1.000000e-03,5.973716e-05,9.318996e-03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.560000e+02,4.190000e+02,2.000000e+00,3.000000e+00,2.302000e+03,8.000000e+01,2010-06-14 11:13:00,1.000000e-03,2.000000e+03,2.490000e+05,3.542435e-01,0.000000e+00,6.233010e-01,0.000000e+00,0.000000e+00,0.000000e+00
50%,4.420000e+02,1.177000e+03,5.000000e+00,6.000000e+00,3.744000e+03,8.000000e+01,2010-06-15 16:56:00,1.000000e-03,1.000000e+04,1.165000e+06,6.600840e-01,9.615385e-02,7.746289e-01,1.000000e+00,0.000000e+00,0.000000e+00
75%,8.450000e+02,7.338000e+03,1.100000e+01,1.000000e+01,1.699300e+04,8.000000e+01,2010-06-16 18:26:00,1.000000e-03,1.400000e+04,4.292000e+06,9.017061e-01,2.500000e-01,9.141816e-01,1.000000e+00,0.000000e+00,0.000000e+00
max,7.632776e+08,1.254005e+09,8.722240e+05,5.147940e+05,6.553500e+04,6.553500e+04,2010-06-17 23:58:00,4.914000e+04,2.802800e+07,2.051278e+10,9.999996e-01,9.999655e-01,9.999982e-01,1.000000e+00,1.000000e+00,1.000000e+00
std,7.517785e+05,1.187177e+06,9.834982e+02,6.679018e+02,2.014297e+04,8.623280e+03,NaN,6.197692e+02,1.000977e+05,8.122032e+07,2.937572e-01,2.021172e-01,2.730417e-01,4.597787e-01,1.447082e-01,1.067566e-01


In [12]:
def calculate_entropy(ip_series):
    counts = ip_series.value_counts(normalize=True)
    return -(counts * np.log2(counts)).sum()

# 5-second window features (existing)
df['time_bucket_5s'] = df.index.floor('5s')
windowed_5s = df.groupby(['destination', 'time_bucket_5s']).agg(
    unique_source_ips_5s=('source', 'nunique'),
    source_ip_entropy_5s=('source', calculate_entropy)
).reset_index()

df = df.reset_index().merge(windowed_5s, on=['destination', 'time_bucket_5s'], how='left')

# 30-second window features
df['time_bucket_30s'] = pd.to_datetime(df['startDateTime']).dt.floor('30s')
windowed_30s = df.groupby(['destination', 'time_bucket_30s']).agg(
    flows_30s=('source', 'count'),
    unique_sources_30s=('source', 'nunique'),
    protocol_diversity_30s=('protocolName', 'nunique'),
    byte_rate_mean_30s=('bytes_per_second', 'mean')
).reset_index()

df = df.merge(windowed_30s, on=['destination', 'time_bucket_30s'], how='left')

# 60-second window features
df['time_bucket_60s'] = pd.to_datetime(df['startDateTime']).dt.floor('60s')
windowed_60s = df.groupby(['destination', 'time_bucket_60s']).agg(
    flows_60s=('source', 'count'),
    unique_sources_60s=('source', 'nunique'),
    protocol_diversity_60s=('protocolName', 'nunique'),
    byte_rate_mean_60s=('bytes_per_second', 'mean')
).reset_index()

df = df.merge(windowed_60s, on=['destination', 'time_bucket_60s'], how='left')

df = df.drop(columns=['time_bucket_5s', 'time_bucket_30s', 'time_bucket_60s'])
df = df.set_index('startDateTime')

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2071657 entries, 2010-06-11 20:10:00 to 2010-06-17 23:58:00
Data columns (total 34 columns):
 #   Column                          Dtype         
---  ------                          -----         
 0   totalSourceBytes                int64         
 1   totalDestinationBytes           int64         
 2   totalDestinationPackets         int64         
 3   totalSourcePackets              int64         
 4   direction                       object        
 5   sourceTCPFlagsDescription       object        
 6   destinationTCPFlagsDescription  object        
 7   source                          object        
 8   protocolName                    object        
 9   sourcePort                      int64         
 10  destination                     object        
 11  destinationPort                 int64         
 12  stopDateTime                    datetime64[ns]
 13  Label                           object        
 14  duration         

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# 1. Extract the date directly from the datetime index
df['date_only'] = df.index.date

# 2. One-hot encode the categorical features BEFORE splitting
df = pd.get_dummies(df, columns=['protocolName', 'direction'], drop_first=True)

X_train_chunks, X_test_chunks = [], []
y_train_chunks, y_test_chunks = [], []

# 3. Group by the exact day (June 14 vs June 15)
for date, daily_data in df.groupby('date_only'):

    # Drop all raw metadata, original counters, identifiers, and the target Label
    # 'startDateTime' is now the index, so it won't be in columns to drop
    X_daily = daily_data.drop(columns=[
        'stopDateTime', 'source', 'destination',
        'sourcePort', 'destinationPort', 'sourceTCPFlagsDescription',
        'destinationTCPFlagsDescription', 'flags', 'duration',
        'totalSourceBytes', 'totalDestinationBytes',
        'totalDestinationPackets', 'totalSourcePackets',
        'Label', 'date_only'
    ], errors='ignore')

    # Convert string labels to binary
    y_daily = daily_data['Label'].apply(lambda x: 0 if x == 'Normal' else 1)

    # Chronological split for THIS day (no shuffling)
    X_tr, X_te, y_tr, y_te = train_test_split(X_daily, y_daily, test_size=0.2, shuffle=False)

    X_train_chunks.append(X_tr)
    X_test_chunks.append(X_te)
    y_train_chunks.append(y_tr)
    y_test_chunks.append(y_te)

# 4. Combine the chunks back into final Train and Test sets
X_train = pd.concat(X_train_chunks)
X_test = pd.concat(X_test_chunks)
y_train = pd.concat(y_train_chunks)
y_test = pd.concat(y_test_chunks)

In [15]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1657323 entries, 2010-06-11 20:10:00 to 2010-06-17 14:07:00
Data columns (total 26 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   packets_per_second      1657323 non-null  float64
 1   bytes_per_second        1657323 non-null  float64
 2   byte_asymmetry          1657323 non-null  float64
 3   packet_asymmetry        1657323 non-null  float64
 4   bidirectional_ratio     1657323 non-null  float64
 5   syn_ratio               1657323 non-null  int64  
 6   rst_ratio               1657323 non-null  int64  
 7   syn_without_data        1657323 non-null  int64  
 8   unique_source_ips_5s    1657323 non-null  int64  
 9   source_ip_entropy_5s    1657323 non-null  float64
 10  flows_30s               1657323 non-null  int64  
 11  unique_sources_30s      1657323 non-null  int64  
 12  protocol_diversity_30s  1657323 non-null  int64  
 13  byte_rate_mean_30s      

In [16]:
import pandas as pd

# Combine X_train and y_train temporarily to undersample safely
train_df = X_train.copy()
train_df['Label'] = y_train

# Separate normal and attack rows in the training set
normal_df = train_df[train_df['Label'] == 0]
attack_df = train_df[train_df['Label'] == 1]

# Aggressive 1:1 undersampling for better attack detection
normal_downsampled = normal_df.sample(n=len(attack_df) * 1)

# Recombine and shuffle just the training partition
balanced_train = pd.concat([normal_downsampled, attack_df]).sample(frac=1)

# Separate back into X_train and y_train
X_train = balanced_train.drop(columns=['Label'])
y_train = balanced_train['Label']

print(f"New training set size: {X_train.shape}")
print(f"Class distribution:\n{y_train.value_counts()}")

New training set size: (99828, 26)
Class distribution:
Label
1    49914
0    49914
Name: count, dtype: int64


In [17]:
X_train.shape

(99828, 26)

In [18]:
y_train.value_counts()

,count
Label,
1,49914
0,49914


In [19]:
y_test.value_counts()

,count
Label,
0,395338
1,18996


In [20]:
import pandas as pd

# Combine X_test and y_test temporarily to undersample safely
test_df = X_test.copy()
test_df['Label'] = y_test

# Separate normal and attack rows in the test set
normal_test_df = test_df[test_df['Label'] == 0]
attack_test_df = test_df[test_df['Label'] == 1]

# Calculate target number of normal samples for ~30% attack
# If attack is 30%, then normal is 70%.
# So, len(attack_test_df) / 0.30 * 0.70 = target_normal_count
n_attack_test = len(attack_test_df)
target_normal_test_count = int((n_attack_test / 0.30) * 0.70)

# Undersample the normal class in the test set
normal_test_downsampled = normal_test_df.sample(n=target_normal_test_count, random_state=42)

# Recombine and shuffle the balanced test set
balanced_test = pd.concat([normal_test_downsampled, attack_test_df]).sample(frac=1, random_state=42)

# Separate back into X_test and y_test
X_test = balanced_test.drop(columns=['Label'])
y_test = balanced_test['Label']

print(f"New test set size: {X_test.shape}")
print(f"Class distribution:\n{y_test.value_counts()}")

New test set size: (63320, 26)
Class distribution:
Label
0    44324
1    18996
Name: count, dtype: int64


In [21]:
X_train.describe()

,packets_per_second,bytes_per_second,byte_asymmetry,packet_asymmetry,bidirectional_ratio,syn_ratio,rst_ratio,syn_without_data,unique_source_ips_5s,source_ip_entropy_5s,flows_30s,unique_sources_30s,protocol_diversity_30s,byte_rate_mean_30s,flows_60s,unique_sources_60s,protocol_diversity_60s,byte_rate_mean_60s
count,9.982800e+04,9.982800e+04,99828.000000,99828.000000,99828.000000,99828.000000,99828.000000,99828.000000,99828.000000,99828.000000,99828.000000,99828.000000,99828.000000,9.982800e+04,99828.000000,99828.000000,99828.000000,9.982800e+04
mean,9.556652e+04,6.166131e+07,0.621141,0.209254,0.742849,0.837270,0.015326,0.113465,5.721150,1.122899,496.210773,5.721150,1.517049,6.029657e+07,496.210773,5.721150,1.517049,6.029657e+07
std,1.419079e+05,9.134410e+07,0.355314,0.202078,0.263022,0.369121,0.122848,0.317162,6.068388,1.323780,539.113587,6.068388,0.506264,8.067862e+07,539.113587,6.068388,0.506264,8.067862e+07
min,6.693440e-05,1.044177e-02,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,-0.000000,1.000000,1.000000,1.000000,1.066667e+00,1.000000,1.000000,1.000000,1.066667e+00
25%,2.000000e+03,2.460000e+05,0.331580,0.000000,0.517047,1.000000,0.000000,0.000000,1.000000,-0.000000,16.000000,1.000000,1.000000,3.824250e+05,16.000000,1.000000,1.000000,3.824250e+05
50%,1.200000e+04,2.209500e+06,0.844198,0.186047,0.868236,1.000000,0.000000,0.000000,1.000000,-0.000000,211.000000,1.000000,2.000000,7.361943e+06,211.000000,1.000000,2.000000,7.361943e+06
75%,2.220000e+05,1.690285e+08,0.899048,0.379447,0.948832,1.000000,0.000000,0.000000,13.000000,2.765927,771.000000,13.000000,2.000000,1.626425e+08,771.000000,13.000000,2.000000,1.626425e+08
max,1.815000e+07,5.842205e+09,0.999994,0.999449,0.999079,1.000000,1.000000,1.000000,79.000000,6.201229,1666.000000,79.000000,3.000000,4.849947e+09,1666.000000,79.000000,3.000000,4.849947e+09


In [22]:
from sklearn.preprocessing import RobustScaler

# Define continuous numerical columns (expanded with new features)
continuous_cols = [
    'packets_per_second', 'bytes_per_second', 'unique_source_ips_5s', 'source_ip_entropy_5s',
    'byte_asymmetry', 'packet_asymmetry', 'bidirectional_ratio',
    'flows_30s', 'unique_sources_30s', 'protocol_diversity_30s', 'byte_rate_mean_30s',
    'flows_60s', 'unique_sources_60s', 'protocol_diversity_60s', 'byte_rate_mean_60s'
]

# Filter to only columns that exist
continuous_cols = [c for c in continuous_cols if c in X_train.columns]

# Initialize and apply RobustScaler
scaler = RobustScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])

print(f"Scaled {len(continuous_cols)} continuous features")
print(X_train[continuous_cols].describe())

Scaled 15 continuous features
       packets_per_second  bytes_per_second  unique_source_ips_5s  \
count        99828.000000      99828.000000          99828.000000   
mean             0.379848          0.352239              0.393429   
std              0.645036          0.541194              0.505699   
min             -0.054545         -0.013091              0.000000   
25%             -0.045455         -0.011633              0.000000   
50%              0.000000          0.000000              0.000000   
75%              0.954545          0.988367              1.000000   
max             82.445455         34.600717              6.500000   

       source_ip_entropy_5s  byte_asymmetry  packet_asymmetry  \
count          99828.000000    99828.000000      99828.000000   
mean               0.405976       -0.393074          0.061161   
std                0.478603        0.626140          0.532559   
min               -0.000000       -1.487657         -0.490310   
25%               -0.00

In [23]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve, f1_score
import matplotlib.pyplot as plt

# Safety net: Sanitize any lingering Infinities
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

# Create validation split from training data (80/20 chronological)
split_idx = int(len(X_train) * 0.8)
X_train_cv = X_train.iloc[:split_idx]
X_val_cv = X_train.iloc[split_idx:]
y_train_cv = y_train.iloc[:split_idx]
y_val_cv = y_train.iloc[split_idx:]

# Calculate dynamic imbalance ratio
imbalance_ratio = (y_train_cv == 0).sum() / (y_train_cv == 1).sum()
print(f"Dynamic imbalance ratio: {imbalance_ratio:.2f}")

# Initialize model with dynamic weights
model = XGBClassifier(
    scale_pos_weight=imbalance_ratio,
    tree_method='hist',
    device='cpu',  # Use CPU; GPU device mismatch fixed
    random_state=42,
    eval_metric='logloss',
    early_stopping_rounds=10,
    n_estimators=500
)

# Train with early stopping on validation set
print("Training model with early stopping...")
model.fit(
    X_train_cv, y_train_cv,
    eval_set=[(X_val_cv, y_val_cv)],
    verbose=50
)

# Generate predictions and probabilities on test set
y_pred = model.predict(X_test)
y_probs = model.predict_proba(X_test)[:, 1]

# Calculate ROC curve for threshold tuning
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

# Find Youden index optimal threshold
youden_idx = np.argmax(tpr - fpr)
youden_threshold = thresholds[youden_idx]

# Calculate precision-recall curve
precision, recall, pr_thresholds = precision_recall_curve(y_test, y_probs)

# Find F1-optimal threshold
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
f1_idx = np.argmax(f1_scores)
f1_threshold = pr_thresholds[f1_idx] if f1_idx < len(pr_thresholds) else 0.5

print(f"\nROC AUC: {roc_auc:.4f}")
print(f"Youden optimal threshold: {youden_threshold:.4f}")
print(f"F1-optimal threshold: {f1_threshold:.4f}")

# Evaluate at default threshold (0.5)
print("\n=== Default Threshold (0.5) ===")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Evaluate at Youden threshold
y_pred_youden = (y_probs >= youden_threshold).astype(int)
print(f"\n=== Youden Threshold ({youden_threshold:.4f}) ===")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_youden))
print("\nClassification Report:\n", classification_report(y_test, y_pred_youden))

# Evaluate at F1 threshold
y_pred_f1 = (y_probs >= f1_threshold).astype(int)
print(f"\n=== F1-Optimal Threshold ({f1_threshold:.4f}) ===")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_f1))
print("\nClassification Report:\n", classification_report(y_test, y_pred_f1))

Dynamic imbalance ratio: 1.00
Training model with early stopping...
[0]	validation_0-logloss:0.44316
[50]	validation_0-logloss:0.01151
[100]	validation_0-logloss:0.01030
[107]	validation_0-logloss:0.01036

ROC AUC: 0.9925
Youden optimal threshold: 0.0123
F1-optimal threshold: 0.0183

=== Default Threshold (0.5) ===
Confusion Matrix:
 [[44214   110]
 [ 3564 15432]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      1.00      0.96     44324
           1       0.99      0.81      0.89     18996

    accuracy                           0.94     63320
   macro avg       0.96      0.90      0.93     63320
weighted avg       0.95      0.94      0.94     63320


=== Youden Threshold (0.0123) ===
Confusion Matrix:
 [[42840  1484]
 [  513 18483]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.97      0.98     44324
           1       0.93      0.97      0.95     18996

    acc

In [24]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Safety net: Sanitize any lingering Infinities
X_train = X_train.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test = X_test.replace([np.inf, -np.inf], np.nan).fillna(0)

# Use a moderated scale_pos_weight (e.g., 5.0 instead of the full extreme ratio)
# to prevent probability compression and massive false-positive spikes.
model = XGBClassifier(
    scale_pos_weight=5.0,
    tree_method='hist',
    device='cuda',
    random_state=42,
    eval_metric='logloss'
)

# Train the model
print("Training model with production-tuned weights...")
model.fit(X_train, y_train)

# Generate raw probabilities
y_probs = model.predict_proba(X_test)[:, 1]

# Apply a realistic, operational threshold (0.3) suitable for a streaming pipeline
production_threshold = 0.1
y_pred_prod = (y_probs >= production_threshold).astype(int)

# Evaluate performance
print("\nProduction-Tuned Confusion Matrix:\n", confusion_matrix(y_test, y_pred_prod))
print("\nProduction-Tuned Classification Report:\n", classification_report(y_test, y_pred_prod))

Training model with production-tuned weights...

Production-Tuned Confusion Matrix:
 [[43853   471]
 [ 2054 16942]]

Production-Tuned Classification Report:
               precision    recall  f1-score   support

           0       0.96      0.99      0.97     44324
           1       0.97      0.89      0.93     18996

    accuracy                           0.96     63320
   macro avg       0.96      0.94      0.95     63320
weighted avg       0.96      0.96      0.96     63320



/usr/local/lib/python3.13/dist-packages/xgboost/core.py:569: UserWarning: [11:44:51] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [25]:
import joblib

# Save the trained XGBoost model with the F1-optimal threshold
model_metadata = {
    'threshold_default': 0.5,
    'threshold_youden': float(youden_threshold),
    'threshold_f1_optimal': float(f1_threshold),
    'roc_auc': float(roc_auc),
    'imbalance_ratio': float(imbalance_ratio),
    'feature_names': list(X_train.columns),
    'training_config': {
        'scale_pos_weight': imbalance_ratio,
        'tree_method': 'hist',
        'random_state': 42,
        'early_stopping_rounds': 10
    }
}

# 1. Save the trained XGBoost model
joblib.dump(model, 'xgboost_ddos_model.pkl')

# 2. Save the fitted RobustScaler for live inference
joblib.dump(scaler, 'robust_scaler.pkl')

# 3. Save metadata for inference pipeline
joblib.dump(model_metadata, 'model_metadata.pkl')

print("✓ Model saved to xgboost_ddos_model.pkl")
print("✓ Scaler saved to robust_scaler.pkl")
print("✓ Metadata saved to model_metadata.pkl")
print(f"\nRecommended Production Threshold: {f1_threshold:.4f} (F1-optimal)")
print(f"Alternative High-Recall Threshold: {youden_threshold:.4f} (Youden)")
print(f"Model ROC AUC: {roc_auc:.4f}")

✓ Model saved to xgboost_ddos_model.pkl
✓ Scaler saved to robust_scaler.pkl
✓ Metadata saved to model_metadata.pkl

Recommended Production Threshold: 0.0183 (F1-optimal)
Alternative High-Recall Threshold: 0.0123 (Youden)
Model ROC AUC: 0.9925
